# 組合せベイズ最適化入門：BOCS・COMBO・Hamming探索

連続変数ではなく、**0または1の組合せ**から最良の条件を選ぶベイズ最適化を学びます。このNotebookは2変数の小さな二値空間で、モデルの学習 → 予測 → 獲得関数による候補提案 → 実測 → 観測更新を順番に示します。

- **BOCS**：二値変数の一次効果と交互作用を二次ベイズ回帰で学習
- **COMBO-style GP**：二値グラフの拡散カーネルで類似度を学習
- **Hamming探索**：1ビットずつ変更する近傍を探索

**実装上の範囲：** BOCSはガウス係数事前分布の実装で、原著のスパース事前分布を完全再現したものではありません。COMBOも二値グラフ拡散GPで、原著の全手順の再現ではありません。


## 0. 二値組合せ空間とは

2変数の二値入力は `(0,0)`, `(1,0)`, `(0,1)`, `(1,1)` の4通りだけです。各変数は**離散的な選択肢**であり、`0.5` のような中間値は許されません。

この例の目的関数は `f(x)=x0+2*x1` で、値を**最大化**します。合成例では真値を計算できますが、実際のBOでは測定前の値は未知です。

In [ ]:
import torch
from robotorchan.models.standard import BOCSSurrogate, COMBOSurrogate
from robotorchan.optim import (
    propose_bocs_candidate,
    propose_combo_candidate,
    run_bocs_sequential,
    run_combo_sequential,
)

torch.manual_seed(0)
dtype = torch.double
train_X = torch.tensor([[0., 0.], [1., 0.]], dtype=dtype)

def objective(X):
    return (X[:, :1] + 2 * X[:, 1:2]).to(dtype=dtype)

train_Y = objective(train_X)
print('Initial:', train_X, train_Y, sep='\n')

### 探索空間と初期観測を確認する

まず全4通りを図示します。**初期観測済みの2点**と未観測の2点を区別し、次にどの組合せを測定するかを考えます。

In [ ]:
import matplotlib.pyplot as plt

all_X = torch.tensor([[0., 0.], [1., 0.], [0., 1.], [1., 1.]], dtype=dtype)
all_Y = objective(all_X)
fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(all_X[:, 0], all_X[:, 1], s=100, label="All binary combinations")
ax.scatter(train_X[:, 0], train_X[:, 1], marker="x", s=130, label="Initially observed")
for x, y in zip(all_X, all_Y):
    ax.annotate(f"f={y.item():.0f}", (x[0].item(), x[1].item()), xytext=(7, 6), textcoords="offset points")
ax.set(xlim=(-0.2, 1.25), ylim=(-0.2, 1.25), xticks=[0, 1], yticks=[0, 1], xlabel="Binary x0", ylabel="Binary x1", title="Four possible configurations")
ax.legend()
plt.show()

## 1. BOCS：二次ベイズ回帰

二値入力の一次・交互作用をモデル化します。ここでの実装はガウス係数事前分布を使用し、原著のスパースBOCSを完全再現するものではありません。

In [ ]:
bocs = BOCSSurrogate(train_X, train_Y)
print('BOCS posterior mean:', bocs.posterior(train_X).mean)
next_bocs = propose_bocs_candidate(train_X, train_Y)
print('BOCS candidate:', next_bocs, 'objective:', objective(next_bocs))

### BOCSの予測平均と不確実性

`BOCSSurrogate` は二値入力から**切片・一次効果・二変数の交互作用**を構成する二次ベイズ回帰です。

BOCSの実装では初期化時に係数事後分布が計算され、明示的な `fit()` も利用できます。ここでは他のモデルと使い方をそろえるために `fit()` を明示します。

`propose_bocs_candidate()` は未観測の二値点を列挙し、BoTorchの `UpperConfidenceBound` を最大化して候補を選びます。**獲得関数値は目的関数の実測値ではありません**。

In [ ]:
bocs.fit()
with torch.no_grad():
    bocs_post = bocs.posterior(all_X.unsqueeze(-2))
    bocs_mean = bocs_post.mean.reshape(-1)
    bocs_sd = bocs_post.variance.clamp_min(0).sqrt().reshape(-1)
fig, ax = plt.subplots(figsize=(8, 4))
labels = ["00", "10", "01", "11"]
ax.errorbar(labels, bocs_mean, yerr=2*bocs_sd, fmt="o", capsize=4, label="BOCS mean ± 2 SD")
ax.scatter(labels, all_Y.squeeze(-1), marker="x", label="True values (demo only)")
ax.set(xlabel="Binary combination (x0,x1)", ylabel="Response", title="BOCS posterior over all combinations")
ax.legend()
plt.show()

## 2. COMBO：グラフ拡散カーネルGP

各二値変数のグラフ構造をカーネルに反映します。候補提案は小規模二値空間の全列挙UCBであり、原著COMBOの全要素を実装するものではありません。

In [ ]:
combo = COMBOSurrogate(train_X, train_Y)
combo.fit()
print('COMBO posterior mean:', combo.posterior(train_X).mean)
next_combo = propose_combo_candidate(train_X, train_Y)
print('COMBO candidate:', next_combo, 'objective:', objective(next_combo))

### COMBOの予測平均と不確実性

COMBO-style GPは、二値変数の**Hammingグラフ上の拡散カーネル**で近い組合せの関係を学習します。

`COMBOSurrogate.fit()` は既存のGP学習APIです。候補提案関数 `propose_combo_candidate()` は内部で新しいCOMBO GPを学習し、未観測の二値候補を全列挙してUCBを最大化します。

この小さな例で候補がBOCSと異なる場合も、1回の提案だけで性能差は判断できません。

In [ ]:
with torch.no_grad():
    combo_post = combo.posterior(all_X.unsqueeze(-2))
    combo_mean = combo_post.mean.reshape(-1)
    combo_sd = combo_post.variance.clamp_min(0).sqrt().reshape(-1)
fig, ax = plt.subplots(figsize=(8, 4))
ax.errorbar(labels, combo_mean, yerr=2*combo_sd, fmt="o", capsize=4, label="COMBO mean ± 2 SD")
ax.scatter(labels, all_Y.squeeze(-1), marker="x", label="True values (demo only)")
ax.set(xlabel="Binary combination (x0,x1)", ylabel="Response", title="COMBO posterior over all combinations")
ax.legend()
plt.show()

### 候補提案と実測値を分ける

`next_bocs` と `next_combo` は**候補の座標**で、実験結果ではありません。上のセルでは学習用の合成関数 `objective()` で候補の値も表示していますが、実際には候補を提案した後で実験装置・シミュレータを実行します。

次は `run_bocs_sequential()` と `run_combo_sequential()` を使い、**提案 → 実測 → 観測追加 → 次の提案**を自動で2回繰り返します。

## 3. 逐次BO：提案 → 評価 → 追加

同一初期データからそれぞれ2回だけ探索します。`objective` は `[1, d]` を受け取り `[1, 1]` を返します。

In [ ]:
bocs_X, bocs_Y = run_bocs_sequential(train_X, train_Y, objective, steps=2)
combo_X, combo_Y = run_combo_sequential(train_X, train_Y, objective, steps=2)
print('BOCS:', bocs_X, bocs_Y.flatten(), sep='\n')
print('COMBO:', combo_X, combo_Y.flatten(), sep='\n')
assert torch.unique(bocs_X, dim=0).shape[0] == 4
assert torch.unique(combo_X, dim=0).shape[0] == 4

### 逐次最適化の観測履歴を比較する

両手法は**同じ初期観測2点**から始め、未観測の2点を追加します。横軸は観測回数、縦軸はその時点までの**最良実測値（best-so-far）**です。

探索空間が4点しかないため、2回追加すると全点を観測します。この図はアルゴリズムの性能優位を示すものではなく、逐次BOの流れを理解するためのものです。

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for name, observed_y in (("BOCS", bocs_Y), ("COMBO", combo_Y)):
    best_so_far = observed_y.squeeze(-1).cummax(dim=0).values
    ax.plot(range(1, len(best_so_far)+1), best_so_far, marker="o", label=name)
ax.set(xlabel="Number of target observations", ylabel="Best observed response", title="Sequential BO learning curves", xticks=[1, 2, 3, 4])
ax.legend()
plt.show()

## 4. Hamming距離と近傍探索

Hamming距離は**2つの組合せで異なるビット数**です。例えば `(0,0)` から `(1,0)` は距離1、`(1,1)` は距離2です。

大規模な組合せ空間では全列挙の費用が指数的に増えます。Hamming近傍探索は**候補探索の方法**であり、BOCS/COMBOのような**代理モデル**とは役割が異なります。

In [ ]:
center = torch.tensor([0., 0.], dtype=dtype)
hamming_distances = (all_X != center).sum(dim=-1)
for x, distance in zip(all_X, hamming_distances):
    print(f"Combination {tuple(int(v) for v in x.tolist())}: Hamming distance = {distance.item()}")
fig, ax = plt.subplots(figsize=(6, 4))
sc = ax.scatter(all_X[:, 0], all_X[:, 1], c=hamming_distances, s=180, cmap="viridis")
ax.set(xlim=(-0.2, 1.2), ylim=(-0.2, 1.2), xticks=[0, 1], yticks=[0, 1], xlabel="x0", ylabel="x1", title="Hamming distance from (0,0)")
fig.colorbar(sc, ax=ax, label="Number of changed bits")
plt.show()

## 4. 比較と制限

- 両方式は二値・単一目的・q=1の例です。
- Hamming Local SearchとDiscrete Trust Regionは獲得関数の候補探索を担当します。状態管理は別途必要です。
- Mixedや制約の対応は手法ごとに異なります。
- 複数シードの比較は `python examples/combinatorial_bo_benchmark.py --seeds 3 --steps 5` を実行してください。
- [理論](../../docs/theory/27_combinatorial_bayesian_optimization.md) / [実装ガイド](../../docs/guide/27_combinatorial_bayesian_optimization.md)

## 5. APIの役割と適用範囲

| 役割 | 既存API | 説明 |
|---|---|---|
| 二次ベイズ回帰 | `BOCSSurrogate` | 二値入力の一次・交互作用を学習 |
| 二値グラフGP | `COMBOSurrogate.fit()` | 拡散カーネルを学習 |
| 次のBOCS候補 | `propose_bocs_candidate()` | 未観測点をUCBで全列挙探索 |
| 次のCOMBO候補 | `propose_combo_candidate()` | GP学習とUCB候補選択 |
| 逐次実験ループ | `run_bocs_sequential()` / `run_combo_sequential()` | 提案・評価・観測追加 |
| 離散近傍探索 | `HammingLocalSearchStrategy` | 獲得関数の局所探索、q=1 |

**制限：** この例は二値・単一目的・q=1です。BOCSとCOMBOの候補提案は小規模空間の全列挙を使用します。Hamming探索と離散Trust Regionは候補探索の別手段であり、代理モデルの学習そのものではありません。

性能比較は `python examples/combinatorial_bo_benchmark.py --seeds 3 --steps 5` を利用してください。詳細は[理論](../../docs/theory/27_combinatorial_bayesian_optimization.md)と[実装ガイド](../../docs/guide/27_combinatorial_bayesian_optimization.md)を参照してください。